In [0]:
# ==========================================
# 📦 IMPORTAÇÕES
# ==========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

In [0]:

# ==========================================
# 📥 CARREGAMENTO DOS DADOS REAIS
# ==========================================
# Fonte pública (GitHub - versão tratada)

url = "https://raw.githubusercontent.com/selva86/datasets/master/GermanCredit.csv"

df = pd.read_csv(url)

# Visual inicial
print("Shape:", df.shape)
print(df.head())

In [0]:
# ==========================================
# 🔍 ENTENDIMENTO DA BASE
# ==========================================
print("\nTipos de dados:")
print(df.dtypes)

print("\nValores nulos:")
print(df.isnull().sum())


In [0]:
# ==========================================
# 🎯 TARGET (RISCO)
# ==========================================
# Risk: good / bad → transformar em binário

df = df.rename(columns={'credit_risk': 'Risk'})

# Garante formato numérico (já está 0 e 1, mas reforçamos)
df['Risk'] = df['Risk'].astype(int)

print("Distribuição do target:")
print(df['Risk'].value_counts())
print("\nTaxa de inadimplência:", df['Risk'].mean())

In [0]:
# ==========================================
# 📊 ESTATÍSTICAS DESCRITIVAS
# ==========================================
print(df.describe())

In [0]:
# ==========================================
# 📈 DISTRIBUIÇÃO DO TARGET
# ==========================================
plt.figure()
sns.countplot(x='Risk', data=df)
plt.title("Distribuição de Inadimplência (Real)")
plt.show()


In [0]:
# ==========================================
# 📊 VARIÁVEIS NUMÉRICAS
# ==========================================
numericas = df.select_dtypes(include=np.number).columns

for col in numericas:
    if col != 'Risk':
        plt.figure()
        sns.histplot(df[col], kde=True)
        plt.title(f"Distribuição de {col}")
        plt.show()

In [0]:
# ==========================================
# 📊 ANÁLISE POR TARGET
# ==========================================
# Comparar comportamento entre bons e maus clientes

for col in numericas:
    if col != 'Risk':
        plt.figure()
        sns.boxplot(x='Risk', y=col, data=df)
        plt.title(f"{col} vs Risco")
        plt.show()


In [0]:
# ==========================================
# 🔗 CORRELAÇÃO
# ==========================================
plt.figure(figsize=(10,8))
sns.heatmap(df[numericas].corr(), annot=True, cmap='coolwarm')
plt.title("Matriz de Correlação")
plt.show()

In [0]:
# ==========================================
# 🧠 CRIAÇÃO DE FEATURE
# ==========================================
# Criando variável relevante para crédito

df['DebtRatio'] = df['amount'] / (df['duration'] + 1)

plt.figure()
sns.boxplot(x='Risk', y='DebtRatio', data=df)
plt.title("Debt Ratio vs Risco")
plt.show()



In [0]:
# ==========================================
# 📊 ANÁLISE CATEGÓRICA
# ==========================================
categoricas = df.select_dtypes(include='object').columns

for col in categoricas:
    plt.figure(figsize=(8,4))
    sns.countplot(x=col, hue='Risk', data=df)
    plt.title(f"{col} vs Risco")
    plt.xticks(rotation=45)
    plt.show()

In [0]:
# ==========================================
# 📊 SEGMENTAÇÃO DE CLIENTES
# ==========================================
df['Faixa_Credito'] = pd.qcut(df['amount'], q=4, labels=['Baixo','Médio','Alto','Muito Alto'])

segmento = df.groupby('Faixa_Credito')['Risk'].mean()

print("\nInadimplência por faixa de crédito:")
print(segmento)

segmento.plot(kind='bar')
plt.title("Risco por Faixa de Crédito")
plt.show()

In [0]:
# ==========================================
# 🚨 OUTLIERS
# ==========================================
for col in numericas:
    if col != 'Risk':
        plt.figure()
        sns.boxplot(y=df[col])
        plt.title(f"Outliers - {col}")
        plt.show()




## 📊 Visão Geral da Base

- **Total de clientes:** 1.000  
- **Quantidade de variáveis:** 21  
- **Valores nulos:** 0 (base íntegra e pronta para análise)  

### 🎯 Distribuição do Target

- **Bons pagadores:** 70% (700 clientes)  
- **Inadimplentes:** 30% (300 clientes)  

> ⚠ A base apresenta **desbalanceamento de classes**, o que deve ser considerado nas etapas de modelagem.

---

## 📉 Inadimplência por Faixa de Crédito

| Faixa de Crédito | Taxa de Inadimplência |
|------------------|----------------------|
| Baixo            | 69,2%               |
| Médio            | 75,2%               |
| Alto             | 77,6%               |
| Muito Alto       | 58,0%               |

> 📌 Observação: A faixa **"Muito Alto" apresenta menor risco relativo**, possivelmente devido a critérios mais rigorosos de concessão ou maior exigência de garantias.

---

## 🔍 Principais Insights

### ⚠ Base desbalanceada
A proporção **70/30** entre bons e maus pagadores pode gerar viés nos modelos preditivos.  
Será necessário aplicar técnicas como:
- Ajuste de pesos (`class_weight`)
- Oversampling (ex: SMOTE)

---

### ⏱ Duração do crédito impacta o risco
Foi identificado um padrão claro:
- Quanto maior o prazo do crédito, maior a probabilidade de inadimplência  

> 📌 Interpretação: prazos longos aumentam a exposição ao risco ao longo do tempo.

---

### ◈ DebtRatio como variável chave
A variável derivada **DebtRatio (valor/duração)** apresentou forte capacidade de separação entre clientes:

- Bons pagadores → menor proporção  
- Maus pagadores → maior proporção  

> 📌 Forte candidata para modelos de risco de crédito.

---

### ⊞ Variáveis categóricas com alto poder preditivo
As seguintes variáveis demonstraram forte relação com inadimplência:

- Status da conta corrente  
- Histórico de crédito  
- Propósito do empréstimo  
- Nível de poupança  

> 📌 Indicam potencial para técnicas como **WOE (Weight of Evidence)**.

---

### ↗ Faixas intermediárias apresentam maior risco
Clientes com valores de crédito **médio e alto** apresentam as maiores taxas de inadimplência (~75% a 78%).

Já clientes com crédito **muito alto** apresentam menor risco relativo (~58%).

> 📌 Possível efeito de seleção:
- Clientes de maior valor tendem a passar por critérios mais rigorosos

---

## 🧠 Conclusão Executiva

A inadimplência está mais associada a **comportamento financeiro e exposição ao risco** do que a variáveis isoladas.

Os principais drivers identificados foram:
- Nível de endividamento
- Duração do crédito
- Histórico financeiro

> 📌 A análise fornece base sólida para construção de um **modelo de score de crédito**, com potencial de aplicação em políticas de concessão e gestão de risco.